In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
import pandas as pd

VIDEO_IDS = [
    "nPUz4DXtYzs", "UyuWysU4v9g", "sPTNYp16nxc", "rSO7gZRDrOM",
    "xSILWfgpfpw", "CGf1qBHETzw", "Q8hUYTE6QUw", "KL9Y0A1pmjE",
    "eVtVg5Nl7ag", "j-oRcnx_beM",
    "AKmBujNy14o", "1CmzeqE3bFw", "2g2QQmCtzjU", "Q3U06YIV_Q0",
    "J8-1vA505jQ", "z_08cYldDco", "1-oom8s1mFg", "JEqaTyGu5u4",
    "HiAMmO4e6pE", "2aiHwEDEUTU"
]

ytt_api = YouTubeTranscriptApi()
all_transcripts = []
for vid in VIDEO_IDS:
    try:
        transcript = ytt_api.fetch(vid, languages=['th', 'en'])
        text = ' '.join([snippet.text for snippet in transcript])
        all_transcripts.append(text)
        print(f"✓ Downloaded: {vid}")
    except Exception as e:
        print(f"✗ Failed: {vid} — {e}")

df = pd.DataFrame({'message': all_transcripts})
print(f"\nTotal transcripts loaded: {len(df)}")
df.head()

✓ Downloaded: nPUz4DXtYzs
✓ Downloaded: UyuWysU4v9g
✓ Downloaded: sPTNYp16nxc
✓ Downloaded: rSO7gZRDrOM
✓ Downloaded: xSILWfgpfpw
✓ Downloaded: CGf1qBHETzw
✓ Downloaded: Q8hUYTE6QUw
✓ Downloaded: KL9Y0A1pmjE
✓ Downloaded: eVtVg5Nl7ag
✓ Downloaded: j-oRcnx_beM
✓ Downloaded: AKmBujNy14o
✓ Downloaded: 1CmzeqE3bFw
✓ Downloaded: 2g2QQmCtzjU
✓ Downloaded: Q3U06YIV_Q0
✓ Downloaded: J8-1vA505jQ
✓ Downloaded: z_08cYldDco
✓ Downloaded: 1-oom8s1mFg
✓ Downloaded: JEqaTyGu5u4
✓ Downloaded: HiAMmO4e6pE
✓ Downloaded: 2aiHwEDEUTU

Total transcripts loaded: 20


,message
0,ทุกคนคือกูมีความคิดจะเปิดข้าวมันไก่เอ้ย เอาเปิ...
1,เรื่องราวมันอยากจะเล่าให้ทุกคนฟังมากเลย ทุกคนแ...
2,สำหรับเนื้อหาหลักของเราในวันนี้นะครับ เราจะมาพ...
3,สำหรับเนื้อหาหลักของเราในวันนี้นะครับ เราจะมาพ...
4,สำหรับเนื้อหาหลักของเราในวันนี้นะครับก็ จะมาเล...


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [4]:
df = pd.read_csv('transcript.csv')
texts = df['message'].astype(str).tolist()

print(f"จำนวนประโยคทั้งหมด: {len(texts)}")
print("ตัวอย่างข้อมูล:", texts[0])

จำนวนประโยคทั้งหมด: 5130
ตัวอย่างข้อมูล: ทุกคน คือ กู มี ความคิด จะ เปิด ข้าวมันไก่ เอ้ย


In [5]:
tokenizer = Tokenizer(filters='')
tokenizer.fit_on_texts(texts)

total_words = len(tokenizer.word_index) + 1
print(f"จำนวนคำศัพท์ทั้งหมด (Vocab size): {total_words}")

input_sequences = []
for line in texts:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"จำนวน sequences ที่สร้างได้: {len(input_sequences)}")

จำนวนคำศัพท์ทั้งหมด (Vocab size): 7879
จำนวน sequences ที่สร้างได้: 140356


In [6]:
max_sequence_len = max([len(x) for x in input_sequences])
print(f"Max Sequence: {max_sequence_len}")

input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

input_sequences = np.array(input_sequences)
X = input_sequences[:,:-1]
y = input_sequences[:,-1]

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

Max Sequence: 552
Shape X: (140356, 551)
Shape y: (140356,)


In [7]:
model = Sequential()

model.add(Embedding(input_dim=total_words, output_dim=100, input_shape=(max_sequence_len-1,)))

model.add(Bidirectional(LSTM(150, return_sequences=False)))
model.add(Dropout(0.2))

model.add(Dense(total_words, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 551, 100)          787900    
                                                                 
 bidirectional (Bidirectiona  (None, 300)              301200    
 l)                                                              
                                                                 
 dropout (Dropout)           (None, 300)               0         
                                                                 
 dense (Dense)               (None, 7879)              2371579   
                                                                 
Total params: 3,460,679
Trainable params: 3,460,679
Non-trainable params: 0
_________________________________________________________________


In [9]:
history = model.fit(X, y, epochs=15, batch_size=64, verbose=1)

Epoch 1/15
2194/2194 [==============================] - 572s 261ms/step - loss: 6.0823 - accuracy: 0.0854
Epoch 2/15
2194/2194 [==============================] - 494s 225ms/step - loss: 5.6278 - accuracy: 0.1245
Epoch 3/15
2194/2194 [==============================] - 512s 234ms/step - loss: 5.3159 - accuracy: 0.1461
Epoch 4/15
2194/2194 [==============================] - 493s 225ms/step - loss: 5.0629 - accuracy: 0.1619
Epoch 5/15
2194/2194 [==============================] - 502s 229ms/step - loss: 4.8378 - accuracy: 0.1749
Epoch 6/15
2194/2194 [==============================] - 478s 218ms/step - loss: 4.6366 - accuracy: 0.1890
Epoch 7/15
2194/2194 [==============================] - 482s 220ms/step - loss: 4.4467 - accuracy: 0.2039
Epoch 8/15
2194/2194 [==============================] - 510s 233ms/step - loss: 4.2742 - accuracy: 0.2193
Epoch 9/15
2194/2194 [==============================] - 514s 234ms/step - loss: 4.1137 - accuracy: 0.2342
Epoch 10/15
2194/2194 [=======================

In [11]:
import pickle
model.save('9arm_model_v1') 

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("บันทึกไฟล์เรียบร้อยแล้ว!")

INFO:tensorflow:Assets written to: 9arm_model_v1\assets


INFO:tensorflow:Assets written to: 9arm_model_v1\assets


บันทึกไฟล์เรียบร้อยแล้ว!


In [17]:
def generate_9arm_text(seed_text, next_words, model, tokenizer, max_sequence_len):
    current_text = seed_text
    
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([current_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break
                
        current_text += " " + output_word
        
    return current_text

test_1 = generate_9arm_text("สวัสดี ครับ", next_words=20, model=model, tokenizer=tokenizer, max_sequence_len=max_sequence_len)
print(f"\n[Test 1] : {test_1}")

test_2 = generate_9arm_text("ตัว ai เนี่ย", next_words=20, model=model, tokenizer=tokenizer, max_sequence_len=max_sequence_len)
print(f"\n[Test 2] : {test_2}")

test_3 = generate_9arm_text("คือ อย่างงี้ เว้ย", next_words=20, model=model, tokenizer=tokenizer, max_sequence_len=max_sequence_len)
print(f"\n[Test 3] : {test_3}")



[Test 1] : สวัสดี ครับ ก็ มี การ ใช้ การ ใช้ ai ใน การจับกุม นะ ครับ ผู้นำ ของ ต่างประเทศ นะ ครับ ก็ คือ เหตุการณ์ ที่

[Test 2] : ตัว ai เนี่ย มัน จะ มี การ ใช้ ai ใน การ ใช้ ai ใน การจับกุม นะ ครับ แล้วก็ มี การ ใช้ ai ใน

[Test 3] : คือ อย่างงี้ เว้ย มึง ก็ ไป นั่ง อ่าน อีเมล กู ก็ ไป เรื่อย เลย นะ กู ก็ แบบ ว่า เฮ้ย กู ก็ แบบ


In [18]:
from tensorflow.keras.models import load_model
import pickle

loaded_model = load_model('9arm_model_v1')

with open('tokenizer.pickle', 'rb') as handle:
    loaded_tokenizer = pickle.load(handle)

print("โหลดโมเดลและ Tokenizer พร้อมใช้งาน!")

โหลดโมเดลและ Tokenizer พร้อมใช้งาน!


In [19]:
test_4 = generate_9arm_text("ทุกคน วันนี้ ผม", next_words=20, model=loaded_model, tokenizer=loaded_tokenizer, max_sequence_len=max_sequence_len)
print(f"\n[Test 3] : {test_4}")


[Test 3] : ทุกคน วันนี้ ผม ก็ จะ ไป ถาม ceo ว่า แบบ ว่า เออ เรา จะ เอา ai ไป ให้ มัน ไป แล้ว อ่ะ มัน
